In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"


In [2]:
os.environ["HOME"] = "/mnt/nas/shuvranshu"
os.environ["HF_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["XDG_CACHE_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.makedirs("/mnt/nas/shuvranshu/huggingface_cache", exist_ok=True)

In [3]:
from langchain_community.document_loaders import   DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import faiss,os
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import numpy as np
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
#hf token 
load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
loader= DirectoryLoader('documents',glob='*.pdf',loader_cls=PyPDFLoader)


In [5]:
#KG implementation
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class Triple:
    subj: str
    rel: str
    obj: str

class SimpleKG:
    def __init__(self):
        self.triples: List[Triple] = []

    def add_triple(self, subj: str, rel: str, obj: str):
        self.triples.append(Triple(subj, rel, obj))

    def find_triples(self, entity: str) -> List[Triple]:
        # return all triples where entity is subject or object
        return [t for t in self.triples if t.subj == entity or t.obj == entity]


KG = SimpleKG()
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_triples_spacy(text):
    doc = nlp(text)
    triples = []
    for token in doc:
        if token.dep_ in ("ROOT", "relcl"):  # verbs or relations
            subj = [w.text for w in token.lefts if w.dep_ in ("nsubj", "nsubjpass")]
            obj = [w.text for w in token.rights if w.dep_ in ("dobj", "pobj", "attr")]
            if subj and obj:
                triples.append((" ".join(subj), token.lemma_, " ".join(obj)))
    return triples




#link entities in query to KG entities
import spacy
nlp = spacy.load("en_core_web_sm")
def extract_entity_mentions(text):
    doc = nlp(text)
    return [ent.text for ent in doc.ents] or [chunk.text for chunk in doc.noun_chunks]

def link_entities(query, kg_entities):
    # simple substring + optional embedding similarity
    mentions = extract_entity_mentions(query)  
    entity_map = {}
    for m in mentions:
        entity_map[m] = [e for e in kg_entities if m.lower() in e.lower()]
    return entity_map

def retrieve_kg_context(query, KG: SimpleKG):
    kg_entities = list(set([t.subj for t in KG.triples] + [t.obj for t in KG.triples]))
    entity_map = link_entities(query, kg_entities)
    triples_text = []
    for ents in entity_map.values():
        for ent in ents:
            for t in KG.find_triples(ent):
                triples_text.append(f"{t.subj} {t.rel} {t.obj}")
    return "\n".join(triples_text)



In [6]:

#combine kg retrieval and context retrieval
def get_combined_context(query, retriever, KG):
    # 1. Retrieve text from FAISS DB
    text_docs = retriever.invoke(query)
    text_context = "\n\n".join([d.page_content for d in text_docs])
    #print(f"context:{text_context}")
    if(len(text_docs)==0):
        return None
    
    # 2. Retrieve KG triples
    kg_context = retrieve_kg_context(query, KG)

    # 3. Combine for final LLM input
    combined_context = f"KG Facts:\n{kg_context}\n\nTextual Context:\n{text_context}"
    return combined_context


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,     
    chunk_overlap=100,   
    separators=["\n\n", "\n", " ", ""]
)

question="Can you tell me about the return policy?" #dummy question for triple extraction
ground_truths=[]
# doc=""
q=0

docs = list(loader.lazy_load())

print("Loaded docs:", len(docs))
print("Sample length:", len(docs[0].page_content))

chunks = text_splitter.split_documents(docs)

print("Total chunks:", len(chunks))

Loaded docs: 2
Sample length: 2232
Total chunks: 7


In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
# import torch

In [9]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   model_kwargs={"device": "cuda"},
                                   encode_kwargs={"normalize_embeddings": True}
                                   )

/tmp/ipykernel_2520637/214096154.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building

In [10]:
#FAISS cosine similarity index
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

dimension = 784


index = faiss.IndexFlatIP(dimension)

In [11]:

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")
print(len(chunks))
print(vectorstore.index.ntotal)

7
7


In [12]:

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
    )

In [ ]:

llm = HuggingFacePipeline.from_model_id(
    model_id="meta-llama/Llama-3.1-8B-Instruct",
    # model_id = "/mnt/nas/shuvranshu/huggingface_cache/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659",
    task="text-generation",
    pipeline_kwargs={
        "temperature": 0.1,
        "max_new_tokens": 256,
        "do_sample": False,
    },
    device_map="auto",
)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2467.38it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [14]:

actor_prompt = PromptTemplate(
    input_variables=["context", "question","feedback_section"],
    template="""
You are a factual assistant whose goal is to produce answers strictly supported by the provided context.



Instructions:
1. Use ONLY the information present in the provided context.
2. Do NOT introduce external knowledge.
3. If the answer cannot be found in the context, don't give any answer.
4. Ensure the answer directly addresses the question.
5. Answer in maximum 3 to 5 words
6. If the answer is implied, infer it.
7. If the answer is Rollo then give no answer at all.


Context:
{context}

Question:
{question}

Refined Answer:
"""
)

def actor(question, context, feedback_section):
    actor_chain = actor_prompt | llm
    # context = get_combined_context(question,retriever, KG)
    response = actor_chain.invoke({
            "context":context,
            "feedback_section": feedback_section,
            "question":question
            }
            )
    answer=response.split('Answer:')[-1].strip()
    return answer


In [ ]:
#evaluator
import json
from langchain_community.chat_models import ChatOllama

In [16]:
evaluator_llm= ChatOllama(model="qwen2.5:7b",
                          format="json",
                          temperature=0)
eval_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a RAG Quality Auditor. You must evaluate ONLY two things:
    1. CONTEXT RELEVANCE: Does the retrieved context contain the information needed to answer the user's query?
    2. FAITHFULNESS: Is the provided answer derived ONLY from the retrieved context?
     
    Output ONLY a JSON object with EXACTLY this structure:
    {{
        "context_relevance": {{
            "score": float (0.0 to 1.0),
            "reason": "string"
        }},
        "faithfulness": {{
            "score": float (0.0 to 1.0),
            "reason": "string"
        }},
        "final_status": "PASS" or "FAIL",
        "action_item": "What should the system do next?(e.g., 'Re-retrieve context' or 'Refine answer').
        if context_relevance score<0.8 then 'Re-retrieve context',if faithfullness score<0.8 then 'Refine answer'.
    }}
    
    Threshold for PASS: Both scores must be >= 0.8.
    """),
    ("user", "USER QUERY: {query}\n\nRETRIEVED CONTEXT: {context}\n\nGENERATED ANSWER: {answer}")
])

/tmp/ipykernel_2520637/2136529005.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  evaluator_llm= ChatOllama(model="qwen2.5:7b",


In [17]:
def evaluate_response(question,context, answer):
    chain = eval_prompt | evaluator_llm
    # response = chain.invoke({"context": context, "answer": answer})
    
    
    try:
        response = chain.invoke({
            "query": question,
            "context": context,
            "answer": answer
        })
        return json.loads(response.content)
    except Exception as e:
        return {
            "final_status": "FAIL",
            "action_item": f"Error in evaluation: {str(e)}"
        }

In [18]:
try:
    response = evaluator_llm.invoke("Hello!")
    print("Ollama is awake and responding!")
except Exception as e:
    print(f"Connection failed. Is the server running? Error: {e}")

Ollama is awake and responding!


In [19]:
query_refiner_llm = ChatOllama(model="qwen2.5:7b", temperature=0)

def generate_refined_query(original_query, critique):
    prompt = f"""
    The initial search for '{original_query}' failed.
    Critique: {critique}
    based on this critique, write a better search query which contains the missing terms or concepts that the original query lacked,
    which led to the failure.
    Don't change the meaning of the question.
    Output ONLY the new search string.
    """
    response = query_refiner_llm.invoke(prompt)
    return response.content.strip()

In [20]:
def run_reflexion_loop(original_question,max_iterations=3):
    
    search_query = original_question
    current_context = get_combined_context(search_query, retriever, KG)
    feedback_section = " " 
    
    for i in range(max_iterations):
        # print(f"\n--- Iteration {i+1} ---")
        
        answer = actor(original_question, current_context, feedback_section)
        
        evaluation = evaluate_response(original_question, current_context, answer)
        # print(json.dumps(evaluation, indent=2))
        # print(current_context)
        # print(f"answer:{answer}")

        if evaluation.get('final_status') == "PASS":
            return answer,current_context
        
        action = evaluation.get('action_item', 'Refine answer')
        
        if action == "Re-retrieve context":
            search_query = generate_refined_query(original_question, evaluation['context_relevance']['reason'])
            # print(f"new search query:{search_query}")
            current_context = get_combined_context(search_query, retriever, KG)
        else:
            feedback_section=evaluation['faithfulness']['reason']
      
    return answer,current_context

In [21]:
# question=qa_pairs[93]["question"]
print(question)
answer,context=run_reflexion_loop(question)
print(context)
print(f"answer:{answer}")
# print(qa_pairs[93]["answer"])

Can you tell me about the return policy?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KG Facts:


Textual Context:
intended for inhalation without combustion 
 
(2) The amount of applicable tax referred to in sub-rule (1) shall be determined in the following manner, 
namely: —

Amendment) Rules, 2025. 
(2)   They shall come into force from 1st day of February, 2026. 
2. In the Central Goods and Services Tax Rules, 2017 (hereinafter referred to as the said rules), after rule 31C, 
the following rule shall be inserted, namely: — 
"31D. Value of supply of goods on basis of retail sale price. -(1) Notwithstanding anything contained in 
the provisions of this Chapter, the value of supply of goods bearing the description specified in column (3), 
falling under the corresponding Chapter/ heading/ sub-heading/ tariff item specified in column (2), of the 
Table below, shall be deemed to be the retail sale price declared on such goods, less the amount of tax as 
applicable, namely: -  
Table 
S. No. Chapter / Heading / 
Sub-heading / 
Tariff item 
Description of Goods

in respect

In [25]:

dataset_ragas = []

dataset_ragas.append({
        "user_input": question,
        "retrieved_contexts": [context],
        "response": answer
    })


In [26]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset_ragas)

In [30]:
from langchain_core.prompts import ChatPromptTemplate
from ragas import evaluate
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.metrics import  Faithfulness, answer_relevancy,context_precision
# answer_relevancy, context_precision

In [35]:
json_prompt = """
You are a strict JSON output generator used for evaluation in RAG systems.
You must always reply ONLY with a valid JSON object — no explanations, no extra text.

If the question asks for a score, output: {"score": <float between 0 and 1>}
If the question asks for a label (e.g., yes/no), output: {"label": "yes"} or {"label": "no"}
Do not include extra keys, comments, or markdown fences.
"""
langchain_llm = ChatOllama(model="qwen2.5:7b",temperature=0,system=json_prompt)



langchain_embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2346.04it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:
result = evaluate(dataset=evaluation_dataset,
                  batch_size=2,
                  metrics=[ Faithfulness(), answer_relevancy],
                  llm=langchain_llm,embeddings=langchain_embeddings)
print(result)

Evaluating: 100%|██████████| 2/2 [00:09<00:00,  4.76s/it]


{'faithfulness': 0.0000, 'answer_relevancy': 0.0000}
